# Sample and null permute portion of every conversation.

Requested by @RickDale

In [1]:
import os
import numpy as np
import pandas as pd
import re
import shutil

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
SERVER_READY_DATA = '../data/server_ready'

OUTPUT_FOLDER = '../data/null-perm'
if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

## Iterate, permute, sample

In [4]:
fs = grab_all_files(SERVER_READY_DATA)
len(fs)

1669

In [5]:
sample_frac = .05

### Everything other than CANDOR

In [6]:
NON_CANDOR = [f for f in fs if (not bool(re.findall(r'CANDOR-\d+-\d+\.parquet', f)))]
len(NON_CANDOR)

13

In [7]:
families = set([f.split('/')[-1].split('-')[0].replace('.parquet', '') for f in NON_CANDOR])
families

{'CABNC',
 'CORAAL',
 'GCSAusE',
 'MICASE',
 'callfriend',
 'grace_corpus',
 'med_school'}

In [ ]:
for fam in tqdm(families):

    familiars = [f for f in NON_CANDOR if (fam in f)]
    df = [
        pq.ParquetFile(f).read(columns=['corpus', 'conversation_id', 'x_turn_id', 'y_turn_id', 'x_speaker', 'y_speaker', 'x_utterance', 'y_utterance']).to_pandas()
        for f in familiars
    ]

    df = pd.concat(df, ignore_index=True)
    if fam == 'grace_corpus':
        df['corpus'] = 'Miao-fNIRS'

    groups = df.groupby(by=['corpus', 'conversation_id'])
    for (corpus, cid), dfi in groups:

        dfi['y_utterance'] = df['y_utterance'].loc[~(
            df['corpus'].isin([corpus]) & df['conversation_id'].isin([cid])

        )].sample(n=len(dfi), replace=((len(df) - len(dfi)) < len(dfi)))

        more_groups, output_data = dfi.groupby(by=['x_turn_id']), []
        for _, dfii in more_groups:
            output_data += [dfii.sample(n=int(np.ceil(sample_frac * len(dfii))))]
        output_data = pd.concat(output_data, ignore_index=True)
        output_data.to_parquet(
            os.path.join(OUTPUT_FOLDER, corpus + '-' + cid + '.parquet'),
            engine='fastparquet',
            compression='snappy'
        )

  0%|          | 0/7 [00:00<?, ?it/s]

### CANDOR Null

In [ ]:
CANDOR =[f for f in fs if bool(re.findall(r'CANDOR-\d+-\d+\.parquet', f))]
len(CANDOR)

In [ ]:
CANDOR = np.random.choice(
    CANDOR,
    size=(184,9),
    replace=False
)

In [ ]:
for i,convoset in enumerate(tqdm(CANDOR)):
    output_ = os.path.join(OUTPUT_FOLDER, 'CANDOR-{}-null_perm.parquet').format(i+1)

    df = pd.concat(
        [pd.read_parquet(f) for f in convoset],
        ignore_index=True
    )

    df['corpus'] = 'CANDOR'

    groups = df.groupby(by=['corpus', 'conversation_id'])
    for (corpus, cid), dfi in groups:

        dfi['y_utterance'] = df['y_utterance'].loc[~(
            df['corpus'].isin([corpus]) & df['conversation_id'].isin([cid])

        )].sample(n=len(dfi), replace=((len(df) - len(dfi)) < len(dfi)))

        more_groups, output_data = dfi.groupby(by=['x_turn_id']), []
        for _, dfii in more_groups:
            output_data += [dfii.sample(n=int(np.ceil(sample_frac * len(dfii))))]

        output_data = pd.concat(output_data, ignore_index=True)

        output_data.to_parquet(
            os.path.join(OUTPUT_FOLDER, corpus + '-' + cid + '.parquet'),
            engine='fastparquet',
            compression='snappy'
        )


## Sort and send

In [ ]:
CANDOR = [f for f in os.listdir(OUTPUT_FOLDER) if ('CANDOR-' in f) and (not f.startswith('.'))]
NON_CANDOR = [f for f in os.listdir(OUTPUT_FOLDER) if ('CANDOR-' not in f) and (not f.startswith('.'))]

In [ ]:
to_rosen, to_itkin = [], []
to_rosen += np.random.choice(CANDOR, size=(int(len(CANDOR)/2),), replace=False).tolist() + np.random.choice(NON_CANDOR, size=(int(len(NON_CANDOR)/2),), replace=False).tolist()
to_itkin = [f for f in CANDOR if f not in to_rosen] + [f for f in NON_CANDOR if f not in to_rosen]
len(to_itkin), len(to_rosen)

In [ ]:
ROSEN_PATH = os.path.join(OUTPUT_FOLDER, 'to_rosen')
if not os.path.exists(ROSEN_PATH):
    os.mkdir(ROSEN_PATH)

for f in tqdm(to_rosen):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ROSEN_PATH, f)
    )

In [ ]:
ITKIN_PATH = os.path.join(OUTPUT_FOLDER, 'to_itkin')
if not os.path.exists(ITKIN_PATH):
    os.mkdir(ITKIN_PATH)

for f in tqdm(to_itkin):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ITKIN_PATH, f)
    )